In [ ]:
from ase.io import read, write
from ase.visualize import view
from mylib.analysis import get_composition, get_unrelaxed_relaxed_energies, calc_surface_area
from mylib.parsers import Parse
import os
from os.path import join
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
import pandas as pd

In [ ]:
MAIN = "./CuGa-reported"
subdirectories = os.listdir(MAIN)
subdirectories.remove("SETTINGS")

In [ ]:
def submain(structure, miller, termination):
    """
    This function reads the structure file and returns the energy, number of elements, and composition.
    """
    path = join(MAIN, structure, miller, termination)
    E_i, E_f = get_unrelaxed_relaxed_energies(path)
    A = calc_surface_area(path)
    composition = get_composition(path)
    out = Parse.outcar(path)
    
    
    return {"structure":structure, "miller":miller, "termination":termination, "E_i": E_i, "E_f": E_f,"A":A, **composition, **out}

In [ ]:
RES =[]
for miller in ["100", "110", "111", "211"]:
    RES.append(submain("Cu3Ga-fcc", miller, "a"))

for miller in ["100", "110"]:
    RES.append(submain("Cu3Ga-fcc", miller, "b"))
df = pd.DataFrame(RES)

In [ ]:
def calc_gamma_corrected(Ef, Ei, A, N, e_bulk):
    return (2*Ef-Ei-N*e_bulk)/(2*A)

def calc_gamma_simple(Ef, A, N, e_bulk):
    return (Ef-N*e_bulk)/(2*A)

In [ ]:
df_bulk = pd.read_csv("../APOTELESMATA/new_formation_energies.csv")

In [ ]:
df

In [ ]:
total = pd.merge(left=df, right=df_bulk.iloc[:, [0,-2]], how="left", on="structure")

In [ ]:
total["gamma_simple"] = 16*calc_gamma_simple(total["E_f"], total["A"], total["N"], total["e"])
total["gamma_corrected"] = 16*calc_gamma_corrected(total["E_f"], total["E_i"], total["A"], total["N"], total["e"])

In [ ]:
total

In [ ]:
total= pd.concat([total, total.loc[[2]]], ignore_index = True)

In [ ]:
total.loc[6, "termination"] = "b"

In [ ]:
total

In [ ]:
total["miller"] = total["miller"].astype(str)

In [ ]:
pure_copper = pd.read_csv("../APOTELESMATA/new_temp.csv")
df = pure_copper[(pure_copper["D3"]==False)].copy()
df["miller"] = df["miller"].astype(str)

In [ ]:
df.to_csv("Copper_surface.csv", index = False)

In [ ]:
total.to_csv("Cu3Ga_surface.csv", index = False)

In [ ]:
fig, ax = plt.subplots(1,1)
sn.scatterplot(df, x="miller", y = "gamma_SI", label = "Copper", color = "k", s = 100)
# sn.scatterplot(df, x="miller", y = "gamma_SI_mp", label = "Copper-mp")
# sn.scatterplot(total, x="miller", y = "gamma_simple", label = "Cu-terminated Cu3Ga relaxed")
sn.scatterplot(total, x="miller", y = "gamma_corrected", hue = "termination", s = 100)

ax.grid()

ax.set(
    title = "Cu3Ga-fcc surfaces (0 K)",
    ylabel = r"$\gamma \quad [ev/\AA^2]$"
)


In [ ]:
fig.savefig("surf_energy.png", dpi = 300)

In [ ]:
total

# Differences

In [ ]:
from ase.data import colors


In [ ]:
def get_diff(path):
    atoms_i = read(join(path, "POSCAR"), format = "vasp")
    atoms_f = read(join(path, "CONTCAR"), format = "vasp")
    fixed_indices = atoms_i.constraints[0].index
    
    Ga_indices = [i for i,_ in enumerate(atoms_i.get_chemical_symbols()) if _=="Ga" and i not in fixed_indices]
    Cu_indices = [i for i,_ in enumerate(atoms_f.get_chemical_symbols()) if _=="Cu" and i not in fixed_indices]
    
    diff =atoms_f.get_positions()-atoms_i.get_positions()
    ga_diff = diff[Ga_indices]
    cu_diff = diff[Cu_indices]
    
    return ga_diff[:, 2], cu_diff[:, 2]

In [ ]:
millers = ["100", "110", "111"]
terminations = ["a", "b"]
main = "./CuGa-reported/Cu3Ga-fcc"
ga_color = colors.jmol_colors[31]
cu_color = colors.jmol_colors[29]

fig, ax = plt.subplots()
shown_ga = False
shown_cu = False

for miller in millers:
    for termination in terminations:
        if miller == "111" and termination == "b":
            continue  # skip if not applicable

        label = f"{miller}-{termination}"
        factor = 100 if termination == "a" else -100
        
        ga, cu = get_diff(join(main, miller, termination))

        x_vals_ga = [label for _ in ga]
        x_vals_cu = [label for _ in cu]

        ax.scatter(x_vals_ga, factor * ga[:], s = 100, alpha =1, color="r",marker="x" ,label="Ga" if not shown_ga else None)
        ax.scatter(x_vals_cu, factor * cu[:], s = 20, alpha =1, color="k",marker="o" ,label="Cu" if not shown_cu else None)
        shown_cu = shown_ga = True


ax.set_xticks(ax.get_xticks())
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.set_ylabel(r"$\Delta z \cdot \hat{n} \quad [pm]$")
ax.set_title("Unrelaxed vs Relaxed difference in the vertical coordinate of the surface.")
ax.set_ylim([-25,25])
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

In [ ]:
fig.savefig("./Surfaces.png", dpi = 300)

# RDF

In [ ]:
from mylib.rdf import compute_partial_rdfs

In [ ]:
def remove_fixed_particles(atoms):
    to_remove = atoms.constraints[0].index
    new = atoms.copy()[[i for i in range(len(atoms)) if i not in to_remove]]
    return new

In [ ]:
atoms_i = read("./CuGa-reported/Cu3Ga-fcc/211/a/POSCAR", format ="vasp")
atoms_f = read("./CuGa-reported/Cu3Ga-fcc/211/a/CONTCAR", format ="vasp")

In [ ]:
new_i = remove_fixed_particles(atoms_i)
new_f = remove_fixed_particles(atoms_f)

In [ ]:
pdf_si = compute_partial_rdfs(new_i, r_max = 5, bin_width=0.01)
pdf_sf = compute_partial_rdfs(new_f, r_max = 5, bin_width=0.01)

In [ ]:
fig, ax1 = plt.subplots(1,1, sharey = True, sharex=True,)

for key, (x,y) in pdf_si.items():
    ax1.plot(x, y, label = f"{key[0]}-{key[1]}")

for key, (x,y) in pdf_sf.items():
    ax1.plot(x, y, linestyle = "-.", c = "grey")
    
ax1.legend()
ax1.set(
    xlim = [2,4]
)

plt.suptitle("Cu3Ga-211")

In [ ]:
fig.savefig("./Cu3Ga_211.png", dpi =300)